# 03 — Pré-processamento dos Dados Meteorológicos

Este notebook realiza o pré-processamento da base meteorológica utilizando **PySpark**, mantendo a proposta de um projeto com foco em **Big Data**.

A base já foi convertida previamente de CSV para Parquet. Neste notebook, os dados serão preparados para a etapa de modelagem preditiva por regressão, cujo objetivo é prever a **temperatura do ar** a partir de variáveis meteorológicas, geográficas e temporais.

A abordagem adotada prioriza:

- processamento distribuído com PySpark;
- preservação de eventos climáticos extremos reais;
- remoção apenas de valores fisicamente inválidos;
- criação de variáveis temporais adequadas para sazonalidade;
- separação temporal entre treino e teste;
- imputação sem vazamento de dados.

## 1. Inicialização da SparkSession

A `SparkSession` é o ponto de entrada para utilização do Spark. Ela permite carregar arquivos, manipular DataFrames e executar transformações distribuídas.

Como este projeto trabalha com volume de dados meteorológicos históricos, o uso de PySpark é mais adequado do que bibliotecas locais como Pandas.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Preprocessamento_Weather_SP_Regressao")
    .getOrCreate()
)

spark

## 2. Importação das bibliotecas

Todas as transformações serão realizadas com funções nativas do PySpark.

Também são utilizadas bibliotecas padrão do Python apenas para normalização textual dos nomes das colunas e para criação das variáveis cíclicas.

In [ ]:
from pyspark.sql.functions import (
    col,
    when,
    count,
    avg,
    min,
    max,
    stddev,
    round,
    regexp_replace,
    regexp_extract,
    to_date,
    year,
    month,
    dayofmonth,
    sin,
    cos,
    lit,
    sum as spark_sum
)

from pyspark.ml.feature import Imputer

import builtins
import unicodedata
import re
import math

## 3. Carregamento da base em Parquet

A base utilizada neste notebook vem do arquivo Parquet gerado no notebook anterior.

O formato Parquet é recomendado em projetos de Big Data porque armazena os dados em formato colunar, o que melhora a performance de leitura, compressão e seleção de colunas.

In [ ]:
input_path = "/home/jovyan/work/data/processed/weather_sp_parquet"

df = spark.read.parquet(input_path)

df.show(5, truncate=False)
df.printSchema()

## 4. Normalização dos nomes das colunas

Os nomes originais das colunas podem conter acentos, espaços, parênteses, vírgulas e outros caracteres especiais.

Para facilitar o uso das colunas em expressões Spark, todos os nomes são padronizados para:

- letras minúsculas;
- sem acentos;
- sem caracteres especiais;
- separação por `_`.

In [ ]:
def normalizar_nome_coluna(nome):
    nome = nome.strip()
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ASCII", "ignore").decode("utf-8")
    nome = nome.lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome)
    nome = nome.strip("_")
    return nome

colunas_normalizadas = [normalizar_nome_coluna(c) for c in df.columns]

df = df.toDF(*colunas_normalizadas)

df.columns

## 5. Renomeação das variáveis principais

Após a normalização automática, algumas colunas ainda ficam com nomes muito longos.

Nesta etapa, as principais variáveis meteorológicas são renomeadas para nomes mais simples e legíveis. Isso melhora a leitura do código e facilita a etapa posterior de modelagem.

In [ ]:
mapa_renomeacao = {
    "precipitacao_total_horario_mm": "precipitacao",
    "pressao_atmosferica_ao_nivel_da_estacao_horaria_mb": "pressao",
    "pressao_atmosferica_max_na_hora_ant_aut_mb": "pressao_maxima",
    "pressao_atmosferica_min_na_hora_ant_aut_mb": "pressao_minima",
    "radiacao_global_kj_m2": "radiacao",
    "temperatura_do_ar_bulbo_seco_horaria_c": "temperatura",
    "umidade_rel_max_na_hora_ant_aut": "umidade_maxima",
    "umidade_rel_min_na_hora_ant_aut": "umidade_minima",
    "umidade_relativa_do_ar_horaria": "umidade",
    "vento_direcao_horaria_gr_gr": "direcao_vento",
    "vento_rajada_maxima_m_s": "rajada_vento",
    "vento_velocidade_horaria_m_s": "velocidade_vento",
    "height": "altitude"
}

for coluna_antiga, coluna_nova in mapa_renomeacao.items():
    if coluna_antiga in df.columns:
        df = df.withColumnRenamed(coluna_antiga, coluna_nova)

df.printSchema()

## 6. Seleção das colunas relevantes

Nesta etapa são mantidas apenas as colunas úteis para a análise climática e para a futura modelagem preditiva.

Foram selecionadas variáveis:

- temporais;
- geográficas;
- meteorológicas;
- identificadoras da estação.

Colunas que não contribuem diretamente para o objetivo inicial são descartadas.

In [ ]:
colunas_uteis = [
    "data",
    "hora",
    "region",
    "state",
    "station",
    "station_code",
    "latitude",
    "longitude",
    "altitude",
    "temperatura",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "radiacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento"
]

colunas_uteis = [c for c in colunas_uteis if c in df.columns]

df = df.select(*colunas_uteis)

df.show(5, truncate=False)
df.printSchema()

## 7. Conversão das colunas numéricas

Algumas colunas numéricas podem ter sido interpretadas como texto, principalmente quando a origem dos dados é CSV.

Nesta etapa, as variáveis meteorológicas e geográficas são convertidas para `double`.

Também é feita a troca de vírgula por ponto, garantindo compatibilidade com o formato decimal esperado pelo Spark.

In [ ]:
colunas_numericas = [
    "latitude",
    "longitude",
    "altitude",
    "temperatura",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "radiacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento"
]

colunas_numericas = [c for c in colunas_numericas if c in df.columns]

for c in colunas_numericas:
    df = df.withColumn(
        c,
        regexp_replace(col(c).cast("string"), ",", ".").cast("double")
    )

df.printSchema()

## 8. Tratamento de valores sentinela

Bases meteorológicas podem utilizar valores como `-9999`, `-999` ou outros números extremamente negativos para representar ausência de medição.

Esses valores não são medições reais. Por isso, devem ser convertidos para `null` antes de qualquer filtro, estatística ou modelagem.

In [ ]:
for c in colunas_numericas:
    df = df.withColumn(
        c,
        when(col(c) <= -999, None).otherwise(col(c))
    )

df.show(5, truncate=False)

## 9. Criação das variáveis temporais

A data e a hora são transformadas em variáveis úteis para análise e modelagem.

São criadas as colunas:

- `data_formatada`;
- `ano`;
- `mes`;
- `dia`;
- `hora_num`.

A conversão da data considera dois formatos possíveis: `yyyy-MM-dd` e `dd/MM/yyyy`.

In [ ]:
df = df.withColumn(
    "data_formatada",
    when(
        to_date(col("data"), "yyyy-MM-dd").isNotNull(),
        to_date(col("data"), "yyyy-MM-dd")
    ).otherwise(
        to_date(col("data"), "dd/MM/yyyy")
    )
)

df = (
    df
    .withColumn("ano", year(col("data_formatada")))
    .withColumn("mes", month(col("data_formatada")))
    .withColumn("dia", dayofmonth(col("data_formatada")))
    .withColumn("hora_num", regexp_extract(col("hora").cast("string"), r"(\d{2})", 1).cast("int"))
)

df.select("data", "data_formatada", "ano", "mes", "dia", "hora", "hora_num").show(20, truncate=False)

## 10. Criação de variáveis temporais cíclicas

Variáveis como mês e hora possuem comportamento cíclico.

Por exemplo:

- dezembro e janeiro estão próximos no ciclo anual;
- 23h e 0h estão próximas no ciclo diário.

Se usarmos apenas `mes = 12` e `mes = 1`, o modelo pode interpretar esses valores como muito distantes. Para evitar isso, são criadas variáveis baseadas em seno e cosseno:

- `mes_sin`;
- `mes_cos`;
- `hora_sin`;
- `hora_cos`.

Essas variáveis ajudam os modelos a capturar padrões sazonais e diários.

In [ ]:
df = (
    df
    .withColumn("mes_sin", sin(2 * lit(math.pi) * col("mes") / lit(12)))
    .withColumn("mes_cos", cos(2 * lit(math.pi) * col("mes") / lit(12)))
    .withColumn("hora_sin", sin(2 * lit(math.pi) * col("hora_num") / lit(24)))
    .withColumn("hora_cos", cos(2 * lit(math.pi) * col("hora_num") / lit(24)))
)

df.select("mes", "mes_sin", "mes_cos", "hora_num", "hora_sin", "hora_cos").show(20, truncate=False)

## 11. Verificação do tamanho inicial da base

Antes dos filtros principais, é importante registrar a quantidade inicial de linhas e colunas.

Isso permite avaliar o impacto das etapas de limpeza.

In [ ]:
linhas_iniciais = df.count()
colunas_iniciais = len(df.columns)

print(f"Linhas iniciais: {linhas_iniciais:,}")
print(f"Colunas iniciais: {colunas_iniciais}")

## 12. Remoção de registros duplicados

Como os dados são horários, espera-se que cada estação tenha apenas uma medição por data e hora.

A chave de duplicidade considera:

- código da estação;
- data formatada;
- hora numérica.

Caso alguma dessas colunas não exista, o código utiliza apenas as disponíveis.

In [ ]:
colunas_chave = [c for c in ["station_code", "data_formatada", "hora_num"] if c in df.columns]

total_antes = df.count()

if len(colunas_chave) > 0:
    df = df.dropDuplicates(colunas_chave)
else:
    df = df.dropDuplicates()

total_depois = df.count()

print(f"Antes: {total_antes:,}")
print(f"Depois: {total_depois:,}")
print(f"Duplicatas removidas: {total_antes - total_depois:,}")

## 13. Aplicação de regras físicas

Nesta etapa são removidos registros com valores incompatíveis com a realidade física das variáveis meteorológicas.

A decisão aqui é importante: **não será aplicada remoção de outliers por IQR**.

Em dados climáticos, valores extremos podem representar eventos reais, como ondas de calor, frentes frias ou períodos de chuva intensa. Remover esses registros por regra estatística genérica poderia apagar justamente fenômenos relevantes para o tema do projeto.

Por isso, serão removidos apenas valores fisicamente inválidos.

In [ ]:
df_tratado = df

if "temperatura" in df_tratado.columns:
    df_tratado = df_tratado.filter(
        col("temperatura").isNull() |
        ((col("temperatura") >= -10) & (col("temperatura") <= 50))
    )

for c in ["umidade", "umidade_maxima", "umidade_minima"]:
    if c in df_tratado.columns:
        df_tratado = df_tratado.filter(
            col(c).isNull() |
            ((col(c) >= 0) & (col(c) <= 100))
        )

for c in ["pressao", "pressao_maxima", "pressao_minima"]:
    if c in df_tratado.columns:
        df_tratado = df_tratado.filter(
            col(c).isNull() |
            ((col(c) >= 800) & (col(c) <= 1100))
        )

for c in ["precipitacao", "radiacao", "velocidade_vento", "rajada_vento"]:
    if c in df_tratado.columns:
        df_tratado = df_tratado.filter(
            col(c).isNull() |
            (col(c) >= 0)
        )

if "direcao_vento" in df_tratado.columns:
    df_tratado = df_tratado.filter(
        col("direcao_vento").isNull() |
        ((col("direcao_vento") >= 0) & (col("direcao_vento") <= 360))
    )

print(f"Antes das regras físicas: {df.count():,}")
print(f"Após regras físicas     : {df_tratado.count():,}")
print(f"Linhas removidas        : {df.count() - df_tratado.count():,}")

## 14. Remoção de registros sem variável-alvo

A variável-alvo do projeto é a temperatura do ar.

Como a proposta é treinar modelos supervisionados de regressão, registros sem valor de temperatura não podem ser usados na modelagem.

Por isso, linhas com `temperatura` nula são removidas.

In [ ]:
df_tratado = df_tratado.filter(col("temperatura").isNotNull())

print(f"Linhas após remover registros sem temperatura: {df_tratado.count():,}")

## 15. Remoção de registros sem variáveis temporais essenciais

Como o projeto analisa a variação da temperatura ao longo do tempo, registros sem `ano`, `mes` ou `hora_num` não são úteis para a modelagem temporal.

Esses registros são removidos antes da criação da base final.

In [ ]:
df_tratado = df_tratado.filter(
    col("ano").isNotNull() &
    col("mes").isNotNull() &
    col("hora_num").isNotNull()
)

print(f"Linhas após remover registros sem variáveis temporais: {df_tratado.count():,}")

## 16. Análise de valores nulos

Após a limpeza inicial, é feita uma contagem de valores nulos por coluna.

Essa análise ajuda a identificar variáveis com excesso de ausência e decidir quais colunas devem ser mantidas na base de modelagem.

In [ ]:
expressoes_nulos = [
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df_tratado.columns
]

df_nulos = df_tratado.select(*expressoes_nulos)

df_nulos.show(truncate=False)

## 17. Percentual de valores nulos por coluna

Além da quantidade absoluta, também é calculado o percentual de valores nulos.

Colunas com grande percentual de ausência podem prejudicar os modelos ou exigir imputações pouco confiáveis.

In [ ]:
total_linhas = df_tratado.count()

expressoes_percentual_nulos = [
    round((count(when(col(c).isNull(), 1)) / total_linhas) * 100, 2).alias(c)
    for c in df_tratado.columns
]

df_percentual_nulos = df_tratado.select(*expressoes_percentual_nulos)

df_percentual_nulos.show(truncate=False)

## 18. Remoção de colunas com excesso de valores nulos

Neste projeto, serão removidas colunas com mais de 40% de valores nulos.

Essa regra evita o uso de variáveis com baixa confiabilidade estatística.

In [ ]:
limite_nulos = 40.0

percentuais = df_percentual_nulos.collect()[0].asDict()

colunas_remover_por_nulos = [
    coluna
    for coluna, percentual in percentuais.items()
    if percentual is not None and percentual > limite_nulos
]

print("Colunas removidas por excesso de nulos:")
print(colunas_remover_por_nulos)

df_tratado = df_tratado.drop(*colunas_remover_por_nulos)

print(f"Quantidade de colunas restantes: {len(df_tratado.columns)}")

## 19. Seleção das variáveis para modelagem

Nesta etapa são selecionadas as variáveis de entrada e a variável-alvo.

A variável-alvo será:

- `temperatura`

As variáveis explicativas incluem informações:

- temporais;
- cíclicas;
- geográficas;
- meteorológicas.

A coluna `ano` é mantida porque ajuda o modelo a capturar tendência temporal de longo prazo.

In [ ]:
features_base = [
    "ano",
    "mes_sin",
    "mes_cos",
    "hora_sin",
    "hora_cos",
    "latitude",
    "longitude",
    "altitude",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "radiacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento"
]

features_base = [c for c in features_base if c in df_tratado.columns]

coluna_alvo = "temperatura"

df_modelagem = df_tratado.select(*(features_base + [coluna_alvo]))

df_modelagem.show(10, truncate=False)
df_modelagem.printSchema()

print("Features selecionadas:")
print(features_base)

## 20. Separação temporal entre treino e teste

Como a base possui natureza temporal, a separação entre treino e teste deve respeitar a ordem cronológica.

Em vez de dividir os dados aleatoriamente, os anos mais antigos são usados para treino e os anos mais recentes para teste.

Essa escolha simula melhor um cenário real, em que o modelo aprende com dados históricos e é avaliado em períodos futuros.

In [ ]:
anos_disponiveis = [
    row["ano"]
    for row in df_modelagem.select("ano")
        .distinct()
        .dropna()
        .orderBy("ano")
        .collect()
]

qtd_anos = len(anos_disponiveis)
qtd_anos_teste = builtins.max(1, int(qtd_anos * 0.2))

anos_teste = anos_disponiveis[-qtd_anos_teste:]
anos_treino = anos_disponiveis[:-qtd_anos_teste]

print("Anos disponíveis:", anos_disponiveis)
print("Anos de treino:", anos_treino)
print("Anos de teste:", anos_teste)

df_treino = df_modelagem.filter(col("ano").isin(anos_treino))
df_teste = df_modelagem.filter(col("ano").isin(anos_teste))

print(f"Linhas de treino antes da imputação: {df_treino.count():,}")
print(f"Linhas de teste antes da imputação : {df_teste.count():,}")

## 21. Imputação de valores ausentes sem vazamento de dados

A imputação deve ser feita com cuidado para evitar vazamento de informação.

Por isso, o imputador é ajustado **apenas com a base de treino**. Depois, a mesma regra aprendida no treino é aplicada tanto no treino quanto no teste.

A estratégia utilizada será a mediana, pois ela é menos sensível a valores extremos do que a média.

In [ ]:
colunas_para_imputar = [
    c for c in features_base
    if c in df_treino.columns
]

imputer = Imputer(
    inputCols=colunas_para_imputar,
    outputCols=[f"{c}_imputado" for c in colunas_para_imputar]
).setStrategy("median")

imputer_model = imputer.fit(df_treino)

df_treino_imputado = imputer_model.transform(df_treino)
df_teste_imputado = imputer_model.transform(df_teste)

features_finais = [f"{c}_imputado" for c in colunas_para_imputar]

df_treino_final = df_treino_imputado.select(*(features_finais + [coluna_alvo]))
df_teste_final = df_teste_imputado.select(*(features_finais + [coluna_alvo]))

df_modelagem_completa = df_treino_final.unionByName(df_teste_final)

print("Features finais após imputação:")
print(features_finais)

print(f"Treino final: {df_treino_final.count():,} linhas")
print(f"Teste final : {df_teste_final.count():,} linhas")
print(f"Base final  : {df_modelagem_completa.count():,} linhas")

## 22. Verificação de nulos após imputação

Após a imputação, a base final deve estar sem valores nulos nas variáveis explicativas e na variável-alvo.

Esta verificação garante que os dados estão prontos para serem utilizados nos modelos de regressão.

In [ ]:
expressoes_nulos_final = [
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df_modelagem_completa.columns
]

df_modelagem_completa.select(*expressoes_nulos_final).show(truncate=False)

## 23. Estatísticas finais da variável-alvo

Antes de salvar os dados, são calculadas estatísticas da variável-alvo `temperatura`.

Essa etapa ajuda a validar se a distribuição final permanece coerente após os tratamentos.

In [ ]:
df_modelagem_completa.select(
    count("*").alias("total_registros"),
    round(avg("temperatura"), 2).alias("media_temperatura"),
    round(min("temperatura"), 2).alias("min_temperatura"),
    round(max("temperatura"), 2).alias("max_temperatura"),
    round(stddev("temperatura"), 2).alias("desvio_temperatura")
).show(truncate=False)

## 24. Salvamento das bases para modelagem

As bases finais são salvas em formato Parquet para serem reutilizadas nos notebooks de modelagem.

São salvas três bases:

- base completa pré-processada;
- base de treino;
- base de teste.

Assim, todos os modelos utilizarão exatamente a mesma preparação e a mesma separação temporal.

In [ ]:
output_path = "/home/jovyan/work/data/processed/weather_sp_preprocessado"
train_path = "/home/jovyan/work/data/processed/weather_sp_train"
test_path = "/home/jovyan/work/data/processed/weather_sp_test"

df_modelagem_completa.write.mode("overwrite").parquet(output_path)
df_treino_final.write.mode("overwrite").parquet(train_path)
df_teste_final.write.mode("overwrite").parquet(test_path)

print(f"Base completa salva em: {output_path}")
print(f"Base de treino salva em: {train_path}")
print(f"Base de teste salva em : {test_path}")

## 25. Criação da base anual para análise climática

Além da base horária para modelagem, também será criada uma base agregada por ano.

Essa base não substitui a base de treino e teste. Ela serve para análise exploratória e storytelling climático, permitindo observar a evolução da temperatura ao longo dos anos.

Serão calculadas estatísticas como:

- temperatura média anual;
- temperatura mínima anual;
- temperatura máxima anual;
- umidade média anual;
- precipitação total anual.

In [ ]:
agregacoes = [
    avg("temperatura").alias("temperatura_media_anual"),
    min("temperatura").alias("temperatura_minima_anual"),
    max("temperatura").alias("temperatura_maxima_anual")
]

if "umidade" in df_tratado.columns:
    agregacoes.append(avg("umidade").alias("umidade_media_anual"))

if "precipitacao" in df_tratado.columns:
    agregacoes.append(spark_sum("precipitacao").alias("precipitacao_total_anual"))

df_tendencia_anual = (
    df_tratado
    .groupBy("ano")
    .agg(*agregacoes)
    .orderBy("ano")
)

df_tendencia_anual.show(30, truncate=False)

## 26. Salvamento da base anual

A base anual será salva separadamente para uso em gráficos e análises de tendência climática.

In [ ]:
tendencia_path = "/home/jovyan/work/data/processed/weather_sp_tendencia_anual"

df_tendencia_anual.write.mode("overwrite").parquet(tendencia_path)

print(f"Base anual de tendência salva em: {tendencia_path}")

## 27. Teste de leitura dos arquivos salvos

Por fim, as bases salvas são lidas novamente para confirmar que o processo de gravação ocorreu corretamente.

In [ ]:
df_base_salva = spark.read.parquet(output_path)
df_treino_check = spark.read.parquet(train_path)
df_teste_check = spark.read.parquet(test_path)
df_tendencia_check = spark.read.parquet(tendencia_path)

print("Base completa:")
df_base_salva.show(5, truncate=False)
print("Linhas:", df_base_salva.count())

print(f"Base de treino: {df_treino_check.count():,} linhas")
print(f"Base de teste : {df_teste_check.count():,} linhas")

print("Base anual de tendência:")
df_tendencia_check.show(10, truncate=False)

## Conclusão do pré-processamento

Neste notebook, a base meteorológica foi preparada em um único pipeline PySpark para uso em modelos de regressão.

As principais decisões adotadas foram:

- tratamento de valores sentinela como nulos;
- remoção apenas de valores fisicamente inválidos;
- preservação de possíveis eventos climáticos extremos;
- criação de variáveis temporais cíclicas para representar sazonalidade;
- separação temporal entre treino e teste;
- imputação ajustada somente na base de treino, evitando vazamento de dados;
- salvamento das bases finais em Parquet.